In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

from optical_setup import OpticalSetupSim2, OpticalSetupSim, OpticalSetup
from mackey_glass import get_mg_train_test_splits2
from lorenz import get_lorenz_train_test_splits2
from utils import validate_data_split_params
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec
from IPython.display import HTML
from reservoir import Reservoir
from LightPipes import cm

In [ ]:
mpl.rcParams.update({
    # Requires a LaTeX install (TeX Live / MiKTeX). Without one, set
    # 'text.usetex': False and 'mathtext.fontset': 'cm' instead.
    "pgf.texsystem"       : "xelatex",
    'text.usetex'         : False,
    'text.latex.preamble' : r'\usepackage{amsmath}',
    'font.family'         : 'serif',   # Computer Modern = default LaTeX font

    # Match your document's font sizes (most journals: 10 pt)
    'font.size'             : 10,
    'axes.labelsize'        : 10,
    'legend.title_fontsize' : 9.2,
    'xtick.labelsize'       : 9,
    'ytick.labelsize'       : 9,
    'legend.fontsize'       : 9,

    # Okabe–Ito palette — colorblind-safe, one line to replace the default cycle
    'axes.prop_cycle': mpl.cycler('color', [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]),

    'lines.linewidth'  : 1.5,
    'axes.linewidth'   : 0.8,
})

okabe_ito = [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]

mm = 1 / 25.4
# fig, ax = plt.subplots(
#     figsize=(85 * mm, 70 * mm),
#     layout='constrained',          # no overlapping labels, ever
# )

# ax.set_xlabel(r'Strain $\varepsilon$ [\%]')
# ax.set_ylabel(r'Stress $\sigma$ [MPa]')
# ax.spines[['top', 'right']].set_visible(False)

# fig.savefig('fig.png', dpi=300, bbox_inches='tight')
# fig.savefig('fig.pdf', bbox_inches='tight')
# fig.savefig('fig.pgf')             # no bbox_inches for PGF
# plt.show()

# Task and Dataset params
---

## MG Chaotic system

In [ ]:
data_split_params = {
    'train_phase': {
        'x0': 1.0,
        'seed': 42,
        'forget': 100, # Get rid of transient states
        'number_train_timesteps': 500, # Must be > forget
        'gap': 0,
    },
    'test_phase': {
        'prediction_horizon': 50, # Must be >= 1
        'number_forecast_origins': 200, # Must be >= 1
        'forecast_origin_spacing': 1, # Always >= 0. It can be different than zero if, and only if number_forecast_origins > 1
        'number_warmup_timesteps': 100 # Must be >= 0. Get rid of transient states
    }
}

forecasting_method = ['multi_step']
res_dim = 512
input_dim = 1

validate_data_split_params(params=data_split_params)

mg, X_train, y_train, X_test_warmup, X_test, y_test = get_mg_train_test_splits2(data_split_params, return_complete_mg=True)

In [ ]:
fig, axs = plt.subplots(
        figsize=(300 * mm, 100 * mm),
        layout='constrained',
    )

axs.set_title(rf"Data split (Initial conditions: {data_split_params['train_phase']['x0']}, seed: {data_split_params['train_phase']['seed']})")

# Train
# axs.plot(mg[0,:], mg[1,:], color='gray', alpha=0.2, marker='.')
axs.plot(X_train[0,:]*0.006, X_train[1,:], marker='.', label='X_train')

# for i in range(data_split_params['test_phase']['prediction_horizon']):
#     axs.plot(X_train[0,:data_split_params['train_phase']['number_train_timesteps']], y_train[:, i], marker='.', label=rf'$y_{{target}}^{{h={i+1}}}$', linestyle=' ')
#     if i == 1:
#         break

# Test
axs.plot(X_test_warmup[0,:]*0.006, X_test_warmup[1,:], marker='.', label='X_test_warmup')
axs.plot(X_test[0,:]*0.006, X_test[1,:], marker='.', label='X_test')

N_o = data_split_params['test_phase']['number_forecast_origins']
s = data_split_params['test_phase']['forecast_origin_spacing'] # s=0 means no spacing (+1 in python)
rolling_window = N_o*(1+s)-s

# for i in range(data_split_params['test_phase']['prediction_horizon']):
#     axs.plot(X_test[0,:rolling_window], y_test[:, i], marker='.', label=rf'$y_{{gt}}^{{h={i+1}}}$', linestyle=' ')
#     if i == 1:
#         break

axs.legend()
axs.set_xlabel(r"Lyapunov time ($\Lambda_{\max} t$)")
axs.set_ylabel(r'Mackey-Glass')
axs.spines[['top', 'right']].set_visible(False)
axs.minorticks_on()
axs.grid(which='major', linestyle='-', alpha=0.2)
axs.grid(which='minor', linestyle='--', alpha=0.1)
axs.set_ylim([0.0, 1.0])
axs.legend()
plt.show()

## Lorenz3D
---

In [ ]:
data_split_params = {
    'train_phase': {
        'x0': [1.0, 1.0, 1.0],
        'seed': 42,
        'forget': 5, # Get rid of transient states
        'number_train_timesteps': 500, # Must be > forget
        'gap': 0,
    },
    'test_phase': {
        'prediction_horizon': 300, # Must be >= 1
        'number_forecast_origins': 1, # Must be >= 1
        'forecast_origin_spacing': 0, # Always >= 0. It can be different than zero if, and only if number_forecast_origins > 1
        'number_warmup_timesteps': 0 # Must be >= 0. Get rid of transient states
    }
}

forecasting_method = ['multi_step', 'one_step']
res_dim = 512
input_dim = 3

validate_data_split_params(params=data_split_params)

lorenz, X_train, y_train, X_test_warmup, X_test, y_test = get_lorenz_train_test_splits2(data_split_params, return_complete_lorenz=True)

## Plot Lorenz3D

In [ ]:
fig = plt.figure(figsize=(18, 10))

gs = GridSpec(3, 2, width_ratios=[2.5, 1])

axs_ts = [fig.add_subplot(gs[i, 0]) for i in range(3)]
ax_3d = fig.add_subplot(gs[:, 1], projection='3d')

var_names = ['x', 'y', 'z']
N_train = data_split_params['train_phase']['number_train_timesteps']
N_o = data_split_params['test_phase']['number_forecast_origins']
s = data_split_params['test_phase']['forecast_origin_spacing']
rolling_window = N_o * (1 + s) - s

for var_idx, ax in enumerate(axs_ts):

    ax.sharex(axs_ts[0])

    ax.set_ylabel(f'{var_names[var_idx]}')

    ax.plot(lorenz[0, :], lorenz[var_idx+1, :], color='gray', alpha=0.2, linestyle='-', marker='.')

    # Train
    ax.plot(X_train[0, :], X_train[var_idx+1, :], marker='.', label='X_train')

    for i in range(data_split_params['test_phase']['prediction_horizon']):
        target_idx = i * 3 + var_idx 
        ax.plot(X_train[0, :N_train], y_train[:, target_idx], marker='.', 
                label=rf'$y_{{target}}^{{h={i+1}}}$', linestyle=' ')
        if i == 1: 
            break

    # Test
    if X_test_warmup.shape[1] > 0:
        ax.plot(X_test_warmup[0, :], X_test_warmup[var_idx+1, :], marker='.', label='X_test_warmup')
    
    ax.plot(X_test[0, :], X_test[var_idx+1, :], marker='.', label='X_test')

    for i in range(data_split_params['test_phase']['prediction_horizon']):
        target_idx = i * 3 + var_idx
        ax.plot(X_test[0, :rolling_window], y_test[:, target_idx], marker='.', 
                label=rf'$y_{{gt}}^{{h={i+1}}}$', linestyle=' ')
        if i == 1: 
            break
            
    # Legenda centralizada acima do primeiro eixo
    if var_idx == 0:
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.25), ncol=6, 
                  frameon=True, facecolor="white", edgecolor="black", framealpha=0.9, fontsize=9)

axs_ts[-1].set_xlabel('Lyapunov time')

# Plotando a Visualização do Atrator 3D
ax_3d.plot(lorenz[1, :], lorenz[2, :], lorenz[3, :], color='gray', alpha=0.3, linewidth=0.5)
ax_3d.plot(X_train[1, :], X_train[2, :], X_train[3, :], marker='.', linestyle='-', color='tab:orange', label='X_train')
ax_3d.plot(X_test[1, :], X_test[2, :], X_test[3, :], marker='.', linestyle='-', color='tab:red', label='X_test')

ax_3d.set_xlabel('X')
ax_3d.set_ylabel('Y')
ax_3d.set_zlabel('Z')
ax_3d.set_title('Attractor')
ax_3d.legend()

plt.tight_layout()
plt.show()

*This is the perfect way to split train and test data. What to notice: at a given time point X(t) (X_train or X_test), the target (y_train or y_test) is X(t+1)*

# Optical Simulation I
---

In [ ]:
optical_setup = optical_setup = OpticalSetupSim(res_dim=res_dim, state_nbin=8, inpt_portion=2.0,
                                device='cpu', seed=7,
                                load_TMs=r'/home/lrvnc/Documents/research/dong-chaotic-systems-fundamentals/data/TMs512nbin8.npz')
optical_setup._init_TM()
# optical_setup._adjust_exposure(linear=0.8, nonlinear=0.4)
# optical_setup._adjust_exposure(linear=0.625, nonlinear=0.16)
optical_setup._adjust_exposure(linear=1.1, nonlinear=1.1)

# Light Pipes
---

In [ ]:
distances = {
    'z_dmd_diff':  20*cm,
    'z_diff_cam_lin':  35*cm,
    'z_diff_dmd_nonlin':  20*cm,
    'z_dmd_cam_nonlin':  20*cm,
}

optical_setup = OpticalSetupSim2(res_dim=res_dim, state_nbin=8, grid_points=512, **distances, save_speckles=True)
optical_setup._adjust_exposure(linear=None, nonlinear=None)

# Real optical setup
---

In [ ]:
optical_setup = OpticalSetup(monitoring=True, grid_points=1024, state_nbin=16, res_dim=res_dim)

In [ ]:
optical_setup._on()

In [ ]:
optical_setup._dmd_warmup(horizon=10, criterion=0.80, period=2)
optical_setup._refresh_ref_speckle() #* Optional

In [ ]:
optical_setup._refresh_ref_speckle() #* Optional
optical_setup._check_stability(optical_path='linear')

In [ ]:
optical_setup._off()

# Training
---

In [ ]:
results = {}

In [ ]:
optical_features = 'linear'
res_type = 'optical'

In [ ]:
optical_setup.reset_speckle_mem()

reservoir = Reservoir(res_dim=res_dim, input_dim=input_dim,
                      data_split_params=data_split_params, forecasting_method=forecasting_method,
                      leaky_rate=0.15, activation_func='norm255', encoding_func='identity',
                      reg_model='ridgeCV', reg_model_params={'fit_intercept': True, 'alphas': np.logspace(-8, 0), 'alpha_per_target': True, 'store_cv_results': False},
                      reg_window=None,
                      res_type=res_type, optical_features=optical_features,
                      optical_setup=optical_setup,
                      seed=42)

# reservoir.init_internal_weights(spectral_radius=0.99)

reservoir.fit(train_data=X_train[1:,:], targets=y_train)
reservoir.predict(warmup_data=X_test_warmup[1:,:], test_data=X_test[1:,:], random_init=False)
# scores = reservoir.score(X_train=X_train[1:,:],
#                          y_test=y_test,
#                          metrics=['nmse_per_dimension_per_horizon', 'running_nmse_per_dimension_per_horizon', 'nmse_per_horizon'])

results[optical_features] = {'y_test': y_test,
                             'reservoir_predictions_multi': reservoir.predictions_multi,
                             'reservoir_predictions_one': reservoir.predictions_one}

if optical_features == 'linear':
    corr = np.array(optical_setup.corr_lin)
elif optical_features == 'nonlinear':
    corr = np.array(optical_setup.corr_nonlin)
else:
    raise ValueError('What kind of magical features are you looking for?')

if (corr > 0.985).all():
    print(f"Correlation min: {corr.min()}. Nice.")
else:
    print(f"Correlation dropped to {corr.min()}. Sad ;-;")

In [ ]:
# 1. Extract data properly to avoid overwriting variables
T = data_split_params['test_phase']['number_forecast_origins']
h_max = data_split_params['test_phase']['prediction_horizon']
input_dim = 3 # Lorenz has 3 dimensions

# Ground Truth (Using linear dict, but it's the same for both)
y_true = results['linear']['y_test'].reshape(T, h_max, input_dim).transpose((2, 0, 1))

# Linear Predictions
y_lin_one = results['linear']['reservoir_predictions_one'].reshape(T, h_max, input_dim).transpose((2, 0, 1))
y_lin_multi = results['linear']['reservoir_predictions_multi'].reshape(T, h_max, input_dim).transpose((2, 0, 1))

# Nonlinear Predictions
y_nonlin_one = results['nonlinear']['reservoir_predictions_one'].reshape(T, h_max, input_dim).transpose((2, 0, 1))
y_nonlin_multi = results['nonlinear']['reservoir_predictions_multi'].reshape(T, h_max, input_dim).transpose((2, 0, 1))

# Setup dimensions
sample_idx = 0
time_steps = np.arange(1, h_max + 1)

# 2. Setup Figure and GridSpec
fig = plt.figure(figsize=(16, 8))
gs = GridSpec(3, 2, width_ratios=[2.5, 1])

axs_ts = [fig.add_subplot(gs[i, 0]) for i in range(3)]
ax_3d = fig.add_subplot(gs[:, 1], projection='3d')

var_names = ['X', 'Y', 'Z']

# Styling dictionaries for cleaner code
styles = {
    'Truth':       {'color': 'black',       'ls': '-',  'marker': '.', 'alpha': 1.0},
    'Lin Single':  {'color': 'tab:blue',    'ls': '--', 'marker': 'x', 'alpha': 0.8},
    'Lin Multi':   {'color': 'tab:cyan',    'ls': '-.', 'marker': '+', 'alpha': 0.8},
    'Nonlin Single':{'color': 'tab:red',     'ls': '--', 'marker': 'x', 'alpha': 0.8},
    'Nonlin Multi': {'color': 'tab:orange',  'ls': '-.', 'marker': '+', 'alpha': 0.8}
}

# 3. Plot Time Series for X, Y, Z
for i, ax in enumerate(axs_ts):
    ax.set_ylabel(f'{var_names[i]}')
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Plot all 5 trajectories
    ax.plot(time_steps, y_true[i, sample_idx, :], label='Ground Truth', **styles['Truth'])
    ax.plot(time_steps, y_lin_one[i, sample_idx, :], label='Linear (Single-Step)', **styles['Lin Single'])
    # ax.plot(time_steps, y_lin_multi[i, sample_idx, :], label='Linear (Multi-Step)', **styles['Lin Multi'])
    ax.plot(time_steps, y_nonlin_one[i, sample_idx, :], label='Nonlinear (Single-Step)', **styles['Nonlin Single'])
    # ax.plot(time_steps, y_nonlin_multi[i, sample_idx, :], label='Nonlinear (Multi-Step)', **styles['Nonlin Multi'])

    # Add legend only to the top subplot
    if i == 0:
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.3), ncol=5, 
                  frameon=True, facecolor="white", edgecolor="black", framealpha=0.9, fontsize=10)

axs_ts[-1].set_xlabel('Prediction Horizon Steps (h)')

# 4. Plot 3D Phase Space
ax_3d.plot(y_true[0, sample_idx, :], y_true[1, sample_idx, :], y_true[2, sample_idx, :],
           label='Ground Truth', **styles['Truth'])

ax_3d.plot(y_lin_one[0, sample_idx, :], y_lin_one[1, sample_idx, :], y_lin_one[2, sample_idx, :],
           label='Lin Single', **styles['Lin Single'])

# ax_3d.plot(y_lin_multi[0, sample_idx, :], y_lin_multi[1, sample_idx, :], y_lin_multi[2, sample_idx, :],
#            label='Lin Multi', **styles['Lin Multi'])

ax_3d.plot(y_nonlin_one[0, sample_idx, :], y_nonlin_one[1, sample_idx, :], y_nonlin_one[2, sample_idx, :],
           label='Nonlin Single', **styles['Nonlin Single'])

# ax_3d.plot(y_nonlin_multi[0, sample_idx, :], y_nonlin_multi[1, sample_idx, :], y_nonlin_multi[2, sample_idx, :],
#            label='Nonlin Multi', **styles['Nonlin Multi'])

ax_3d.set_xlabel('X')
ax_3d.set_ylabel('Y')
ax_3d.set_zlabel('Z')
ax_3d.set_title('3D Phase Space Trajectories', fontsize=14)
ax_3d.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.cumsum((y_test[0,:].flatten() - reservoir.predictions_one[0,:].flatten())**2 ) / (X_train[1,:].var() * np.arange(1, 1001)))